之前的学习主要围绕stream()，invoke()方法，这里补充一些batch调用的示例

- batch()特点是等待所有请求处理完毕，按原始输入顺序返回结果列表
- batch_as_completed()特点是在每个请求处理完毕后立即返回结果，按请求完成顺序返回结果列表


In [1]:
from dotenv import load_dotenv
import os
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from rich import print as rich_print
load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    model="gpt-5.4-mini",
)
print(model)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}} output_version=None profile={'name': 'GPT-5.4 mini', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000018E22CE78C0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000018E22E44590> root_client=<openai.OpenAI object at 0x0000018E2

## 对比一下batch()和循环invoke()

In [2]:
# 准备多个输入
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]
# ✅ 批量调用（高效）
import time
start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start
print("批量调用结果：")
for i, response in enumerate(responses): # 遍历每个响应
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")

批量调用结果：
1. “Spring has arrived.”
2. “It’s very hot in summer.”
3. “秋天落叶” in English is **“autumn leaves”** or **“falling leaves in autumn”** depending on context.
4. “冬天下雪” can be translated into English as:

**It snows in winter.**
耗时: 1.48秒



In [3]:
# ❌ 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]
start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)

loop_time = time.time() - start
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"循环调用耗时: {loop_time:.2f}秒")
print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. “Spring has arrived.”
2. “It’s very hot in summer.”
3. “秋天落叶” in English is **“autumn leaves”** or **“falling leaves in autumn”** depending on context.
4. “冬天下雪” can be translated into English as:

**It snows in winter.**
循环调用耗时: 3.89秒
批量调用节省: 62.0%


## 使用场景

- 用AI生成测试用例